In [ ]:
import pandas as pd 
import numpy as np
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt 
%matplotlib inline
from scipy.stats  import gaussian_kde
import plotly.graph_objects as go


In [ ]:
df = pd.read_csv("data/train.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
df.duplicated()

In [ ]:
df.isnull().sum()

In [ ]:
df.isna().sum()

In [ ]:
columns = ['tenure','monthlycharges','totalcharges']

for col in columns:
    print(df[col].skew())

In [ ]:
columns = ['tenure','monthlycharges','totalcharges']

for col in columns:
    fig = px.histogram(
        data_frame=df,
        x=col,
        nbins=30,
        title=f"Distribution of {col}"
    )

    fig.update_layout(
        xaxis_title=col,
        yaxis_title="Frequency"
    )

    fig.show()

In [ ]:
columns = ['tenure','monthlycharges','totalcharges']

for col in columns:

    data = df[col].dropna()
    kde = gaussian_kde(data)

    x_values = np.linspace(
        data.min(),
        data.max(),
        500
    )

    y_values = kde(x_values)

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=y_values,
            mode="lines",
            name="KDE"
        )
    )

    fig.update_layout(
        title= f"KDE Distribution of {col}",
        xaxis_title=col,
        yaxis_title="Density"
    )

    fig.show()



In [ ]:
columns = ['tenure','monthlycharges','totalcharges']

fig = go.Figure()


for col in columns:

    fig.add_trace(
        go.Box(
            y=df[col],
            name=col
        )
    )

    fig.update_layout(
        title="Boxplot comparision of Numerical Features",
        yaxis_title="value"
    )

    fig.show()

    

In [ ]:
outlier_summary = []
columns = ['tenure','monthlycharges','totalcharges']

for col in columns:
     q1 = df[col].quantile(0.25)
     q3 = df[col].quantile(0.75)

     iqr = q3-q1

     lower_bound = q1 - 1.5 * iqr
     upper_bound = q3 + 1.5 * iqr

     outliers = df[
          (df[col] < lower_bound) | (df[col] > upper_bound)
     ]

     outlier_summary = [{
          "feature":col,
          "q1":q1,
          "q3":q3,
          "iqr":iqr,
          "lower_bound":lower_bound,
          "upper_bound":upper_bound,
          "outlier_count":len(outliers),
          "outlier%": len(outliers) / len(df) * 100
     }]

     outlier_df = pd.DataFrame(outlier_summary)
     print(outlier_df)



In [ ]:
df.head(2)

In [ ]:
df['churn'].value_counts()

In [ ]:
df['churn'].isnull().sum()

In [ ]:
cat_columns = df.select_dtypes(
    include=['object','category']

).columns.tolist()

print(cat_columns)

In [ ]:
cat_features = [
    col for col in cat_columns
    if col != 'churn'
]

print(cat_features)

In [ ]:
for col in cat_features:
    print(f"{col}: {df[col].nunique()} unique values")

In [ ]:
for col in cat_features:
    print(df[col].value_counts())

In [ ]:
for col in cat_features:
    percentages = (
        df[col]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )

    print(percentages)

In [ ]:
for col in cat_features:
    counts = (
        df[col]
        .value_counts()
        .reset_index()
    )
    counts.columns = [col,'count']

    fig = px.bar(
        counts,
        x=col,
        y='count',
        text = 'count',
        title=f"distribution of {col}"
    )

    fig.update_layout(
        xaxis_title=col,
        yaxis_title = "number of customers"
    )

    fig.show()

In [ ]:
# detect rare categories
for col in cat_features:
    percentages = (
        df[col]
        .value_counts(normalize=True)
        .mul(100)
    )

    print(percentages[percentages<1])

In [ ]:
cat_miss = df[cat_features].isnull().sum()
cat_miss = cat_miss[
    cat_miss > 0
]

print(cat_miss)

In [ ]:
for col in cat_features:
    print(
        df[col]
        .astype(str)
        .str.strip()
        .value_counts()
    )

In [ ]:
contract_churn = pd.crosstab(
    df['contract'],
    df['churn'],
    normalize='index'
) *100

contract_churn = contract_churn.reset_index()
contract_churn

In [ ]:
fig = px.bar(
    contract_churn,
    x='contract',
    y=['No','Yes'],
    title='churn distribution by contract',
    barmode='stack'
)

fig.update_layout(
    xaxis_title='contract type',
    yaxis_title='percentage (%)'
)

fig.show()

In [ ]:
def cat_churn_analysis(df,column):
    churn_rate=(
        df.groupby(column)['churn']
        .apply(lambda x: (x == 'Yes').mean() * 100)
        .reset_index(name='churn_rate')
    )

    fig = px.bar(
        churn_rate,
        x=column,
        y='churn_rate',
        text=churn_rate['churn_rate'].round(2),
        title=f"churn rate by {column}"
    )

    fig.update_layout(
        xaxis_title=column,
        yaxis_title='churn_rate (%)'
    )

    fig.show()

    return churn_rate

In [ ]:
for col in cat_features:
    result = cat_churn_analysis(df,col)
    print(result)

In [ ]:
num_features = ['tenure','monthlycharges','totalcharges']

In [ ]:
corr_matrix = df[num_features].corr()
print (corr_matrix)

fig = px.imshow(
    corr_matrix,
    text_auto=True,
    title='correlation matrix of numerical features',
    aspect='auto'
)

fig.update_layout(
    xaxis_title='features',
    yaxis_title='features'
)

fig.show()

In [ ]:
miss = df.isnull().sum()
miss = miss[miss > 0]
print(miss)

In [ ]:
miss_percent = (
    df.isnull().mean() * 100
)
miss_percent = miss_percent[
    miss_percent > 0
]

print(miss_percent)